### DELTA LIVE TABLES - GOLD LAYER

**COACHES DLT PIPELINE**


In [0]:
# A multi-step streaming transformation where:
# Raw streaming data is ingested into a DLT streaming table.
# An intermediate view applies transformations (e.g. filtering, cleaning).
# A final DLT streaming table consumes that view and possibly applies aggregations or writes it out.

In [0]:
import dlt
from pyspark.sql.functions import *

### Expectations for Data Quality

In [0]:
expec_coaches = {
        "rule1": "code is not null",
        "rule2": "current is True"
    }

In [0]:
expec_nocs = {
        "rule1": "code is not null"
    }

In [0]:
expec_events = {
        "rule1": "event is not null"
    }

In [0]:
@dlt.table

def source_coaches():
    df = spark.readStream.table("olympics.silver.coaches")
    return df

In [0]:
@dlt.view

def view_coaches():

    df = spark.readStream.table("LIVE.source_coaches")
    df = df.fillna("Unknown")
    return df
    

In [0]:
@dlt.table

@dlt.expect_all(expec_coaches)
def coaches():

    df = spark.readStream.table("LIVE.view_coaches")
    return df

**NOCS DLT PIPELINE**

In [0]:
@dlt.view

def source_nocs():

    df = spark.readStream.table("olympics.silver.nocs")
    return df

In [0]:
@dlt.table

@dlt.expect_all_or_drop(expec_nocs)
def nocs():

    df = spark.readStream.table("LIVE.source_nocs")
    return df

**EVENTS DLT PIPELINE**

In [0]:
@dlt.view

def source_events():
    
    df = spark.readStream.table("olympics.silver.events")
    return df

In [0]:
@dlt.table

@dlt.expect_all(expec_events)
def events():

    df = spark.readStream.table("LIVE.source_events")
    return df

### CDC - Apply Changes (DLT)

In [0]:
# To build a Delta Live Tables (DLT) pipeline for master data management of athletes, using:
# A streaming view on the athletes master table.
# Change Data Capture (CDC) logic using the apply_changes API on empty streaming table.

In [0]:
@dlt.view

def source_athletes():
    df = spark.readStream.table("olympics.silver.athletes")
    return df

In [0]:
dlt.create_streaming_table("athletes")

In [0]:
dlt.apply_changes(
    target = "athletes",
    source = "source_athletes",
    keys = ["athlete_id"],
    sequence_by = "height",
    stored_as_scd_type=1
)